# ColumnTransformer

In this notebook, we will learn:
- Why we need `ColumnTransformer`
- How preprocessing was done before it
- The practical use of `ColumnTransformer`
- A clear, step-by-step code example with model training

## 1) Why do we need ColumnTransformer?

Real-world datasets usually have mixed column types:
- Numerical columns (age, salary, marks)
- Categorical columns (city, gender, product type)

Different preprocessing is needed for different columns:
- Numerical -> scaling (like `StandardScaler`)
- Categorical -> encoding (like `OneHotEncoder`)

`ColumnTransformer` lets us apply the right transformer to the right columns in one unified step. This prevents confusion and keeps preprocessing consistent for train/test/new data.

## 2) How was it done before ColumnTransformer?

Earlier, we often did preprocessing manually:
1. Select categorical columns and apply `pd.get_dummies()`
2. Manually scale numerical columns
3. Make sure train and test have same columns
4. Handle unseen categories manually

This approach works for small demos, but for real projects it is error-prone and hard to maintain.

In [ ]:
import pandas as pd

# Small mixed-type dataset (numerical + categorical)
df = pd.DataFrame({
    "age": [25, 32, 47, 51, 62, 23, 40, 36],
    "salary": [30000, 50000, 70000, 75000, 90000, 28000, 62000, 52000],
    "city": ["Delhi", "Mumbai", "Delhi", "Pune", "Mumbai", "Pune", "Delhi", "Mumbai"],
    "gender": ["M", "F", "F", "M", "M", "F", "M", "F"],
    "bought": [0, 1, 1, 1, 1, 0, 1, 0]
})

df.head()

In [ ]:
# Manual old approach (for understanding only)
X_manual = df.drop("bought", axis=1)
y = df["bought"]

# 1) Encode categorical columns manually
X_manual_encoded = pd.get_dummies(X_manual, columns=["city", "gender"], drop_first=True)

# 2) Manual scaling (example)
for col in ["age", "salary"]:
    mean = X_manual_encoded[col].mean()
    std = X_manual_encoded[col].std()
    X_manual_encoded[col] = (X_manual_encoded[col] - mean) / std

print("Manual preprocessed shape:", X_manual_encoded.shape)
X_manual_encoded.head()

## 3) ColumnTransformer approach (recommended)

Now we apply:
- `StandardScaler` on numerical columns
- `OneHotEncoder` on categorical columns

And combine preprocessing + model in one `Pipeline`.

This is cleaner, safer, and production-friendly.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Features and target
X = df.drop("bought", axis=1)
y = df["bought"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Define column groups
numeric_features = ["age", "salary"]
categorical_features = ["city", "gender"]

# Create preprocessing block
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Full pipeline: preprocessing + model
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Optional: see transformed feature names
preprocessor_fitted = model.named_steps["preprocessor"]
feature_names = preprocessor_fitted.get_feature_names_out()

print("Total transformed features:", len(feature_names))
print(feature_names)

## 4) Final use of ColumnTransformer

`ColumnTransformer` is useful because it:
- Applies different preprocessing to different column types in one place
- Avoids manual mistakes in train/test preprocessing
- Works smoothly with `Pipeline` and model training
- Is easier to deploy in real ML workflows

In short: it makes preprocessing organized, repeatable, and reliable.